# 16 · Réseaux feed-forward — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch16_feedforward/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch16_feedforward/03_notebook.ipynb`, créée par `python tools/start_chapter.py 16`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 16.Q1 à 16.Q10, 16.R1 à 16.R3, 16.1 à 16.5, 16.7, 16.8 | vérifier tes réponses courtes | 🧠 🔁 ✏️ ∂ 🧮 | ★ à ★★★ | |
| A | 16.11 | Prédire les formes des sorties avant d'exécuter | 🔮 | ★ | 10 |
|  | 16.12 | Compter les paramètres par programme : count_parameters | 🔨 | ★ | 10 |
|  | 16.13 | La couche dense : dense_forward | 🔨 | ★ | 15 |
| B | 16.14 | Dix façons d'initialiser : init_weights | 🔨 | ★★ | 25 |
|  | 16.15 | Reproduire les figures 16.5 et 16.6, puis comparer Glorot et He | 🎨 | ★★ | 20 |
|  | 16.16 | Ordre topologique d'un graphe : topological_order | 🔨 | ★★ | 25 |
| C | 16.17 | Empiler les couches : init_mlp et mlp_forward | 🔨 | ★★ | 30 |
|  | 16.18 | Relire un MLPClassifier de scikit-learn avec ton mlp_forward | 📦 | ★★ | 25 |
|  | 16.19 | Initialisation constante : les neurones restent-ils jumeaux ? | 🔮 | ★★ | 20 |
|  | 16.20 | Chasse au bug : transposée, biais mal diffusé, variance prise pour un écart-type | 🐛 | ★★ | 25 |
|  | 16.21 | Rendre l'aléatoire reproductible et le prouver avec pytest | 🛠️ | ★★ | 20 |
| D | 16.22 | Lire des histogrammes d'activations couche par couche | 📈 | ★★ | 20 |
|  | 16.23 | Activations à travers 20 couches selon l'initialisation | 🔬 | ★★★ | 45 |
|  | 16.24 | Défi : garder vivant un réseau ReLU de 50 couches | 🏆 | ★★★ | 60 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="16")  # your own library

## Objectifs et rappel express

- Suivre la forme de chaque tableau d'une passe avant matricielle, et compter les paramètres d'un réseau dense.
- Écrire et vérifier contre PyTorch, scikit-learn et la bibliothèque standard les fonctions de `mylearn.nn.layers` : `count_parameters`, `dense_forward`, `init_weights`, `topological_order`, `init_mlp` et `mlp_forward`.
- Voir pourquoi une initialisation constante ne marche pas, et ce que changent LeCun, Glorot et He.
- Diagnostiquer, couche par couche, des activations qui s'évanouissent, explosent ou saturent.

**Rappel express.** Une couche : $\mathbf{Z} = \mathbf{X}\mathbf{W} + \mathbf{b}$, avec $\mathbf{X}$ $(n, n_{in})$, $\mathbf{W}$ $(n_{in}, n_{out})$, $\mathbf{b}$ $(n_{out},)$ ; un MLP : $\mathbf{A}^{(\ell)} = f(\mathbf{A}^{(\ell-1)}\mathbf{W}^{(\ell)} + \mathbf{b}^{(\ell)})$, $\mathbf{A}^{(0)} = \mathbf{X}$ ; paramètres : $P = \sum_\ell (n_{\ell-1} n_\ell + n_\ell)$ ; variance d'une somme pondérée : $\mathrm{Var}(z) = n_{in}\,\mathrm{Var}(w)\,\mathbb{E}[x^2]$ ; LeCun $\mathrm{Var}(w) = 1/n_{in}$, Glorot $2/(n_{in} + n_{out})$, He $2/n_{in}$ ; $\mathcal{U}(-a, a)$ a l'écart-type $a/\sqrt{3}$. En NumPy : `X @ W + b`, `rng = np.random.default_rng(seed)`, `rng.normal(0.0, std, size)`, `rng.uniform(-a, a, size)` ; en PyTorch, `nn.Linear(n_in, n_out)` range ses poids en `(n_out, n_in)`.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 16.1 à 16.5, ∂ 16.7 et 🧮 16.8)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; des nœuds ou une suite de codes : les lettres collées, dans l'ordre (`"XYZ"`) ; une forme NumPy : entre guillemets (`"(4, 7)"`, `"(7,)"`), ou `"erreur"` ; plusieurs nombres : une liste (`[4, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la preuve ∂ 16.6, les démonstrations de ∂ 16.7, la réflexion (🗣️ 16.9, 📄 16.10) et l'entretien se corrigent avec `05_solutions.md`.

**16.Q1 — Qu'est-ce qui rend un réseau « feed-forward » ?**

In [ ]:
# 16.Q1: the values you found on paper
answer_16_Q1a = ...  # a) True or False
answer_16_Q1b = ...  # b) the letter of your choice, e.g. "E"
answer_16_Q1c = ...  # c) True or False
answer_16_Q1d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_16_Q1a, answer_16_Q1b, answer_16_Q1c, answer_16_Q1d]):
    wb.check(f"16.Q1{letter}", answer)

**16.Q2 — Vocabulaire des graphes : nœuds, arêtes, parents, descendants**

In [ ]:
# 16.Q2: the values you found on paper
answer_16_Q2a = ...  # a) the parents, letters in alphabetical order, e.g. "XY"
answer_16_Q2b = ...  # b) the descendants, letters in alphabetical order
answer_16_Q2c = ...  # c) a whole number
answer_16_Q2d = ...  # d) True or False
answer_16_Q2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_16_Q2a, answer_16_Q2b, answer_16_Q2c, answer_16_Q2d, answer_16_Q2e]):
    wb.check(f"16.Q2{letter}", answer)

**16.Q3 — Vrai ou faux : la passe arrière crée une boucle de rétroaction**

In [ ]:
# 16.Q3: the values you found on paper
answer_16_Q3a = ...  # a) True or False
answer_16_Q3b = ...  # b) True or False
answer_16_Q3c = ...  # c) the letter of your choice, e.g. "E"

for letter, answer in zip("abc", [answer_16_Q3a, answer_16_Q3b, answer_16_Q3c]):
    wb.check(f"16.Q3{letter}", answer)

**16.Q4 — Pourquoi des graphes sans boucle, et où est l'exception des RNN ?**

In [ ]:
# 16.Q4: the values you found on paper
answer_16_Q4a = ...  # a) True or False
answer_16_Q4b = ...  # b) the letter of your choice, e.g. "E"
answer_16_Q4c = ...  # c) True or False
answer_16_Q4d = ...  # d) a whole number

for letter, answer in zip("abcd", [answer_16_Q4a, answer_16_Q4b, answer_16_Q4c, answer_16_Q4d]):
    wb.check(f"16.Q4{letter}", answer)

**16.Q5 — Réseau synchrone ou asynchrone : reconnaître les deux**

In [ ]:
# 16.Q5: the values you found on paper
answer_16_Q5a = ...  # a) four letters S or A, e.g. "SSSS"
answer_16_Q5b = ...  # b) True or False
answer_16_Q5c = ...  # c) True or False

for letter, answer in zip("abc", [answer_16_Q5a, answer_16_Q5b, answer_16_Q5c]):
    wb.check(f"16.Q5{letter}", answer)

**16.Q6 — Horodatage : quand un nœud doit-il recalculer sa sortie ?**

In [ ]:
# 16.Q6: the values you found on paper
answer_16_Q6a = ...  # a) True or False
answer_16_Q6b = ...  # b) True or False
answer_16_Q6c = ...  # c) True or False
answer_16_Q6d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_16_Q6a, answer_16_Q6b, answer_16_Q6c, answer_16_Q6d]):
    wb.check(f"16.Q6{letter}", answer)

**16.Q7 — Dans un programme, qui garantit l'ordre d'évaluation ?**

In [ ]:
# 16.Q7: the values you found on paper
answer_16_Q7a = ...  # a) the letter of your choice, e.g. "E"
answer_16_Q7b = ...  # b) True or False
answer_16_Q7c = ...  # c) True or False

for letter, answer in zip("abc", [answer_16_Q7a, answer_16_Q7b, answer_16_Q7c]):
    wb.check(f"16.Q7{letter}", answer)

**16.Q8 — Un neurone, trois enfants : combien de poids ?**

In [ ]:
# 16.Q8: the values you found on paper
answer_16_Q8a = ...  # a) a whole number
answer_16_Q8b = ...  # b) True or False
answer_16_Q8c = ...  # c) a whole number
answer_16_Q8d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_16_Q8a, answer_16_Q8b, answer_16_Q8c, answer_16_Q8d]):
    wb.check(f"16.Q8{letter}", answer)

**16.Q9 — Pourquoi pas le même poids partout au départ ?**

In [ ]:
# 16.Q9: the values you found on paper
answer_16_Q9a = ...  # a) True or False
answer_16_Q9b = ...  # b) the letter of your choice, e.g. "E"
answer_16_Q9c = ...  # c) True or False
answer_16_Q9d = ...  # d) a number with 3 decimals

for letter, answer in zip("abcd", [answer_16_Q9a, answer_16_Q9b, answer_16_Q9c, answer_16_Q9d]):
    wb.check(f"16.Q9{letter}", answer)

**16.Q10 — LeCun, Glorot, He : quel fan-in, pour quelle activation ?**

In [ ]:
# 16.Q10: the values you found on paper
answer_16_Q10a = ...  # a) a number with 3 decimals
answer_16_Q10b = ...  # b) a number with 3 decimals
answer_16_Q10c = ...  # c) the letter of your choice, e.g. "E"
answer_16_Q10d = ...  # d) the letter of your choice, e.g. "E"
answer_16_Q10e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_16_Q10a, answer_16_Q10b, answer_16_Q10c, answer_16_Q10d, answer_16_Q10e]):
    wb.check(f"16.Q10{letter}", answer)

**16.R1 — Que contient un estimateur après fit() ?**

In [ ]:
# 16.R1: the values you found on paper
answer_16_R1a = ...  # a) True or False
answer_16_R1b = ...  # b) the letter of your choice, e.g. "E"
answer_16_R1c = ...  # c) a whole number
answer_16_R1d = ...  # d) a shape between quotes, e.g. "(4, 7)", or "erreur"
answer_16_R1e = ...  # e) a shape between quotes, e.g. "(4, 7)", or "erreur"

for letter, answer in zip("abcde", [answer_16_R1a, answer_16_R1b, answer_16_R1c, answer_16_R1d, answer_16_R1e]):
    wb.check(f"16.R1{letter}", answer)

**16.R2 — Combien de régions pour un arbre de profondeur 2 ?**

In [ ]:
# 16.R2: the values you found on paper
answer_16_R2a = ...  # a) a whole number
answer_16_R2b = ...  # b) a whole number
answer_16_R2c = ...  # c) True or False
answer_16_R2d = ...  # d) a whole number

for letter, answer in zip("abcd", [answer_16_R2a, answer_16_R2b, answer_16_R2c, answer_16_R2d]):
    wb.check(f"16.R2{letter}", answer)

**16.R3 — Formes compatibles dans un produit matriciel**

In [ ]:
# 16.R3: the values you found on paper
answer_16_R3a = ...  # a) a shape between quotes, e.g. "(4, 7)", or "erreur"
answer_16_R3b = ...  # b) a shape between quotes, e.g. "(4, 7)", or "erreur"
answer_16_R3c = ...  # c) a shape between quotes, e.g. "(4, 7)", or "erreur"
answer_16_R3d = ...  # d) a whole number
answer_16_R3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_16_R3a, answer_16_R3b, answer_16_R3c, answer_16_R3d, answer_16_R3e]):
    wb.check(f"16.R3{letter}", answer)

**Ex 16.1 — Compter les poids et les biais d'un MLP**

In [ ]:
# Ex 16.1: the values you found on paper
answer_16_1a = ...  # a) a whole number
answer_16_1b = ...  # b) a whole number
answer_16_1c = ...  # c) a whole number
answer_16_1d = ...  # d) a whole number
answer_16_1e = ...  # e) a whole number
answer_16_1f = ...  # f) a whole number
answer_16_1g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_16_1a, answer_16_1b, answer_16_1c, answer_16_1d, answer_16_1e, answer_16_1f, answer_16_1g]):
    wb.check(f"16.1{letter}", answer)

**Ex 16.2 — Passe avant à la main dans un réseau 2-2-1**

In [ ]:
# Ex 16.2: the values you found on paper
answer_16_2a = ...  # a) a number
answer_16_2b = ...  # b) a list [z1, z2]
answer_16_2c = ...  # c) a number
answer_16_2d = ...  # d) a number with 3 decimals
answer_16_2e = ...  # e) a number
answer_16_2f = ...  # f) True or False
answer_16_2g = ...  # g) a list of two numbers

for letter, answer in zip("abcdefg", [answer_16_2a, answer_16_2b, answer_16_2c, answer_16_2d, answer_16_2e, answer_16_2f, answer_16_2g]):
    wb.check(f"16.2{letter}", answer)

**Ex 16.3 — Formes et broadcasting d'un mini-batch à travers les couches**

In [ ]:
# Ex 16.3: the values you found on paper
answer_16_3a = ...  # a) a shape between quotes, e.g. "(4, 7)", or "erreur"
answer_16_3b = ...  # b) a shape between quotes, e.g. "(4, 7)", or "erreur"
answer_16_3c = ...  # c) a shape between quotes, e.g. "(4, 7)", or "erreur"
answer_16_3d = ...  # d) a shape between quotes, e.g. "(4, 7)", or "erreur"
answer_16_3e = ...  # e) a shape between quotes, e.g. "(4, 7)", or "erreur"
answer_16_3f = ...  # f) a whole number
answer_16_3g = ...  # g) a shape between quotes, e.g. "(4, 7)", or "erreur"
answer_16_3h = ...  # h) a shape between quotes, e.g. "(4, 7)", or "erreur"
answer_16_3i = ...  # i) a shape between quotes, e.g. "(4, 7)", or "erreur"

for letter, answer in zip("abcdefghi", [answer_16_3a, answer_16_3b, answer_16_3c, answer_16_3d, answer_16_3e, answer_16_3f, answer_16_3g, answer_16_3h, answer_16_3i]):
    wb.check(f"16.3{letter}", answer)

**Ex 16.4 — Ordre d'évaluation d'un graphe et détection de boucle**

In [ ]:
# Ex 16.4: the values you found on paper
answer_16_4a = ...  # a) the nodes in order, letters glued
answer_16_4b = ...  # b) a whole number
answer_16_4c = ...  # c) a whole number
answer_16_4d = ...  # d) True or False
answer_16_4e = ...  # e) True or False
answer_16_4f = ...  # f) a whole number

for letter, answer in zip("abcdef", [answer_16_4a, answer_16_4b, answer_16_4c, answer_16_4d, answer_16_4e, answer_16_4f]):
    wb.check(f"16.4{letter}", answer)

**Ex 16.5 — Simuler un réseau asynchrone avec horodatage**

In [ ]:
# Ex 16.5: the values you found on paper
answer_16_5a = ...  # a) a number
answer_16_5b = ...  # b) a number
answer_16_5c = ...  # c) a whole number
answer_16_5d = ...  # d) a whole number
answer_16_5e = ...  # e) a number

for letter, answer in zip("abcde", [answer_16_5a, answer_16_5b, answer_16_5c, answer_16_5d, answer_16_5e]):
    wb.check(f"16.5{letter}", answer)

**Ex 16.7 — Conserver la variance : retrouver les règles de LeCun, Glorot et He**

In [ ]:
# Ex 16.7: the values you found on paper
answer_16_7a = ...  # a) a number
answer_16_7b = ...  # b) a number with 4 decimals
answer_16_7c = ...  # c) a number
answer_16_7d = ...  # d) a number with 4 decimals
answer_16_7e = ...  # e) a number with 3 decimals
answer_16_7f = ...  # f) a number with 1 decimal
answer_16_7g = ...  # g) a number with 6 decimals

for letter, answer in zip("abcdefg", [answer_16_7a, answer_16_7b, answer_16_7c, answer_16_7d, answer_16_7e, answer_16_7f, answer_16_7g]):
    wb.check(f"16.7{letter}", answer)

**Ex 16.8 — Fermi : taille, mémoire et coût d'un MLP pour MNIST**

In [ ]:
# Ex 16.8: the values you found on paper
answer_16_8a = ...  # a) a number of MB, 1 decimal
answer_16_8b = ...  # b) a number of millions, 2 decimals
answer_16_8c = ...  # c) a whole number of billions
answer_16_8d = ...  # d) a whole number of seconds
answer_16_8e = ...  # e) a whole number of seconds
answer_16_8f = ...  # f) a number of MB, 1 decimal

for letter, answer in zip("abcdef", [answer_16_8a, answer_16_8b, answer_16_8c, answer_16_8d, answer_16_8e, answer_16_8f]):
    wb.check(f"16.8{letter}", answer)

## Partie A · Une couche dense

Prédire des formes (16.11), compter les paramètres d'un réseau (16.12), puis la passe avant d'une couche (16.13). La cellule suivante définit les outils des parties A à D et charge 2 000 images MNIST aplaties (784 pixels bruts, de 0 à 255) ; `X_STD` les standardise avec **une** moyenne et **un** écart-type pour tous les pixels, comme le suppose la fiche (§16.4.1). Ne recopie jamais un nombre à la main : chaque réponse est calculée par ton code, et pas arrondie. Après un redémarrage du noyau, exécute la cellule de configuration, puis la cellule d'outils qui suit (`# Tools for the notebook exercises`), avant toute vérification.

In [ ]:
# Tools for the notebook exercises (parts A to D), and a sample of MNIST
import contextlib
import inspect
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
from wb.errors import MylearnMissingError

try:
    import torch
except ImportError:                       # the comparisons with PyTorch are then skipped
    torch = None


def verdict(ex_id, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message."""
    print(f"✅ Ex {ex_id} : {success}" if ok else f"❌ Ex {ex_id} : {failure}")


def filled(*values):
    """True when none of the values is still `...` (or None): the answer has been written."""
    return all(value is not ... and value is not None for value in values)


def fr(x, digits=3):
    """A number written the French way, with a decimal comma (for the messages in French)."""
    return f"{x:.{digits}f}".replace(".", ",")


def returned(ex_id, name, value):
    """False, with a ❌ message, when the function `name` returned None (a forgotten return); True otherwise."""
    if value is None:
        print(f"❌ Ex {ex_id} : {name} renvoie None : as-tu oublié le return ?")
        return False
    return True


def print_answer(ex_id, value, **_):
    """Solutions notebook: show the value that the exercise notebook checks with wb.check."""
    try:
        shown = np.round(np.asarray(value, dtype=float), 4).tolist() if isinstance(value, (np.ndarray, list, tuple)) else value
    except (TypeError, ValueError):
        shown = value
    print(f"{ex_id}:", shown)


@contextlib.contextmanager
def checked(ex_id):
    """Like wb.attempt (⏳ while the code is not written yet), but any other error raised by YOUR code becomes one ❌
    line instead of stopping the cell, so that the tests run after it can say more."""
    with wb.attempt(ex_id):
        try:
            yield
        except (NotImplementedError, ImportError, MylearnMissingError):
            raise
        except Exception as exc:                              # a bug in the learner's code: report it, go on
            if isinstance(exc, TypeError) and "NoneType" in str(exc):
                raise                                         # wb.attempt explains the forgotten return
            hint = " ; une fonction ou une méthode oublie-t-elle son return ?" if "NoneType" in str(exc) else ""
            print(f"❌ Ex {ex_id} : ton code lève {type(exc).__name__} : {str(exc)[:300]}{hint}")


def written(probe):
    """False while probe() stops on code not written yet (NotImplementedError, or no mylearn at all); True otherwise,
    even when it raises another error (the tests then say which)."""
    try:
        probe()
    except (NotImplementedError, ImportError, MylearnMissingError):
        return False
    except Exception:
        return True
    return True


def shape_of(compute):
    """The shape of compute() written as in the exercises, e.g. "(64, 128)", or "erreur" if it raises."""
    try:
        return str(tuple(np.shape(compute())))
    except Exception:
        return "erreur"


def relu(z):
    """ReLU, used as a black box until chapter 17."""
    return np.maximum(0.0, z)


TEST_FILES = {"layers": "tests/test_ch16_layers.py"}


def run_mylearn_tests(module, keyword, impl="learner"):
    """Run the tests of mylearn.nn.<module> selected by `keyword` (on YOUR code by default)."""
    test_file = TEST_FILES[module]
    command = [sys.executable, "-m", "pytest", test_file, "-k", keyword, "-q", "-p", "no:cacheprovider",
               "--color=no", "-rf", "--tb=no"]
    if impl != "learner":
        command.append(f"--impl={impl}")
    try:
        result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True, timeout=300,
                                env={**os.environ, "COLUMNS": "1000"})   # long lines: the reason of each failure
    except subprocess.TimeoutExpired:
        print("❌ les tests n'ont pas fini en 5 min : une boucle de ton code ne s'arrête-t-elle pas ?")
        return
    lines = result.stdout.strip().splitlines()
    failed = [line for line in lines if line.startswith("FAILED ")]
    if any("'NoneType' object has no attribute" in line for line in failed):
        print("💡 un attribut est lu sur None : une fonction oublie-t-elle son return ?")
    for line in failed[:8]:                                  # the test, then the reason of its failure
        name, _, reason = line.removeprefix(f"FAILED {test_file}::").partition(" - ")
        print(f"❌ {name}\n   {reason[:800]}")
    if len(failed) > 8:
        print(f"   ... and {len(failed) - 8} other failed test(s)")
    print("pytest:", lines[-1] if lines else result.stderr.strip()[-300:])


X_MNIST, Y_MNIST = wb.datasets.load_mnist("train", n=2000, flatten=True, seed=1616)   # raw pixels 0..255 (uint8)
X_STD = (X_MNIST.astype(float) - X_MNIST.mean()) / X_MNIST.std()                        # one mean, one std for all
print(X_MNIST.shape, X_MNIST.dtype, "· standardized:", round(float(X_STD.mean()), 3), round(float(X_STD.std()), 3))

### Ex 16.11 — Prédire les formes des sorties avant d'exécuter 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir la forme de chaque tableau d'une passe avant, et les opérations qui lèvent une erreur ou n'en lèvent pas.  
**Prérequis :** ✏️ 16.3 · 0A (broadcasting) · fiche, section sur la passe avant · **Fil rouge :** MNIST · **Parcours :** C

Une couche de 128 neurones reçoit des images MNIST standardisées : `X_11` est un mini-batch de 64 images (forme `(64, 784)`), `X128_11` un mini-batch de 128 images, `W1_11` a la forme `(784, 128)`, `b1_11` la forme `(128,)` et `W2_11`, les poids d'une couche suivante de 10 neurones, la forme `(128, 10)`.

Prédis, **avant** d'exécuter quoi que ce soit, la forme du résultat de chaque expression, entre guillemets (par exemple `"(3, 5)"` ou `"(7,)"`), ou `"erreur"` si NumPy lève une erreur :
a) `shape_a_11` : `X_11 @ W1_11 + b1_11` ;
b) `shape_b_11` : `X_11[0] @ W1_11` ;
c) `shape_c_11` : `X_11 @ W1_11.T` ;
d) `shape_d_11` : `X_11 @ W1_11 + b1_11.reshape(-1, 1)` ;
e) `shape_e_11` : `np.tanh(X_11[:1] @ W1_11) @ W2_11` ;
f) `shape_f_11` : `X128_11 @ W1_11 + b1_11.reshape(-1, 1)`.

Puis exécute l'expérience : elle calcule chaque expression, affiche ta prédiction et ce que donne NumPy, et vérifie tes réponses.

In [ ]:
rng_11 = np.random.default_rng(1611)
X_11 = X_STD[:64]                               # a mini-batch of 64 images
X128_11 = X_STD[:128]                           # a mini-batch of 128 images
W1_11 = rng_11.normal(0, 0.05, size=(784, 128))
b1_11 = np.zeros(128)
W2_11 = rng_11.normal(0, 0.1, size=(128, 10))

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
shape_a_11 = ...   # a) "(…)" or "erreur"
shape_b_11 = ...   # b)
shape_c_11 = ...   # c)
shape_d_11 = ...   # d)
shape_e_11 = ...   # e)
shape_f_11 = ...   # f)

In [ ]:
if any(answer is ... or answer is None for answer in [shape_a_11, shape_b_11, shape_c_11, shape_d_11, shape_e_11, shape_f_11]):
    print("⏳ Ex 16.11 : écris d'abord tes six prédictions, puis relance cette cellule.")
else:
    found_11 = {"a": shape_of(lambda: X_11 @ W1_11 + b1_11),
                "b": shape_of(lambda: X_11[0] @ W1_11),
                "c": shape_of(lambda: X_11 @ W1_11.T),
                "d": shape_of(lambda: X_11 @ W1_11 + b1_11.reshape(-1, 1)),
                "e": shape_of(lambda: np.tanh(X_11[:1] @ W1_11) @ W2_11),
                "f": shape_of(lambda: X128_11 @ W1_11 + b1_11.reshape(-1, 1))}
    for letter, prediction in zip("abcdef", [shape_a_11, shape_b_11, shape_c_11, shape_d_11, shape_e_11, shape_f_11]):
        print(f"{letter}) you predicted {prediction!r:12} · NumPy gives {found_11[letter]!r}")
    wb.check("16.11a", shape_a_11)
    wb.check("16.11b", shape_b_11)
    wb.check("16.11c", shape_c_11)
    wb.check("16.11d", shape_d_11)
    wb.check("16.11e", shape_e_11)
    wb.check("16.11f", shape_f_11)

Dans tes notes : pourquoi d) lève-t-il une erreur et pas f) ? Que contient la case `[i, k]` du résultat de f), et pourquoi est-ce un bug silencieux ? Quelle précaution prendrais-tu dans un test pour qu'il ne passe pas inaperçu ?

### Ex 16.12 — Compter les paramètres par programme : count_parameters 🔨 ★ ⏱️ 10 min
**Objectif :** écrire la formule du nombre de paramètres d'un réseau dense, avec ses cas d'erreur.  
**Prérequis :** ✏️ 16.1 · fiche, section sur la passe avant (compter les paramètres) · **Fil rouge :** synthétique · **mylearn :** `nn/layers.py` · **Parcours :** R, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/nn/layers.py` (créé par `python tools/start_chapter.py 16` ; c'est le premier module du sous-package `mylearn.nn`), lis la docstring de chaque fonction, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy seulement ; PyTorch, scikit-learn et la bibliothèque standard sont les **oracles** des tests. La cellule de vérification recharge ta librairie, essaie les exemples de la docstring, compare quelques résultats à un oracle, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch16_layers.py -q` les lance tous dans un terminal (la docstring du module dit `pytest tests/ -q` : préfère cette commande du chapitre, plus courte et plus lisible).

Écris `count_parameters`. **Contrat** (la docstring, plus ces précisions que vérifient les tests) : `count_parameters(layer_sizes, bias=True)` reçoit une liste ou un tuple de tailles `[n_0, n_1, ..., n_L]` (des entiers Python ou NumPy, au moins 2, tous ≥ 1) et rend un **entier Python** (`int`, pas `np.int64`) : la somme, sur les couches, de $n_{\ell-1} n_\ell$ poids, plus $n_\ell$ biais si `bias` est vrai ; `ValueError` pour moins de 2 tailles, une taille < 1, un booléen ou un nombre non entier.

La vérification essaie les exemples de la docstring, compare ton compte à celui de `torch.nn.Linear` sur trois réseaux, puis lance les tests.

Dans tes notes : pourquoi refuser `True` comme taille, alors que Python accepte `True + 1` ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="16")   # reload your saved file
with checked("16.12"):
    doc_12 = [mylearn.nn.layers.count_parameters([2, 4, 4, 1]), mylearn.nn.layers.count_parameters([2, 4, 4, 1], bias=False),
              mylearn.nn.layers.count_parameters([784, 256, 128, 10])]
    print("docstring examples:", doc_12, "(expected [37, 28, 235146])")
with checked("16.12"):
    for sizes_12 in ([3, 5, 2], [784, 512, 512, 10], [10, 1]):
        mine_12 = mylearn.nn.layers.count_parameters(sizes_12)
        if torch is not None:
            layers_12 = [torch.nn.Linear(a, b) for a, b in zip(sizes_12[:-1], sizes_12[1:])]
            oracle_12 = sum(p.numel() for layer in layers_12 for p in layer.parameters())
            verdict("16.12", mine_12 == oracle_12, f"{sizes_12} : {mine_12} paramètres, comme torch.nn.Linear.",
                    f"{sizes_12} : PyTorch compte {oracle_12} paramètres, toi {mine_12}.")
        else:
            print(f"{sizes_12}: {mine_12} parameters (PyTorch absent: the PyTorch oracle test is skipped, run it on Colab; "
                  "the other tests still check your counts)")
if written(lambda: mylearn.nn.layers.count_parameters([2, 1])):
    run_mylearn_tests("layers", keyword="test_count_parameters_")
else:
    print("⏳ Ex 16.12 : les tests attendent ton count_parameters.")

### Ex 16.13 — La couche dense : dense_forward 🔨 ★ ⏱️ 15 min
**Objectif :** écrire la passe avant d'une couche dense sur un mini-batch, avec ses vérifications de forme.  
**Prérequis :** 0A (broadcasting) · 0B (produit matriciel) · ch. 10 (une couche en une ligne) · fiche, section sur la passe avant · **Fil rouge :** synthétique · **mylearn :** `nn/layers.py` · **Parcours :** R, M, C

> **mylearn** : dans `mon_travail/mylearn/nn/layers.py`, mêmes règles qu'en 16.12. Enregistre, puis relance la cellule de vérification.

Écris `dense_forward`. **Contrat** (la docstring, plus ces précisions que vérifient les tests) : `dense_forward(x, W, b=None)` convertit `x` en tableau `float64` (une liste de listes est acceptée), une ligne par exemple, ou un seul exemple en 1-D ; `W` a la forme `(n_in, n_out)` ; `b` est un vecteur `(n_out,)` ou `None` (pas de biais). Elle rend `z = x @ W + b` en `float64`, de forme `(n, n_out)`, ou `(n_out,)` pour un `x` 1-D ; chaque ligne ne dépend que de la ligne correspondante de `x`. `ValueError` si `W` n'est pas 2-D, si `x` n'est ni 1-D ni 2-D, si `x.shape[-1] != W.shape[0]` ou si `b.shape != (W.shape[1],)` (un biais en colonne `(n_out, 1)` est refusé, même quand NumPy saurait l'additionner). Elle ne modifie pas ses arguments.

La vérification essaie les exemples de la docstring, compare ta couche à `torch.nn.functional.linear` sur 50 images, vérifie qu'une image seule donne la même sortie que dans le batch, puis lance les tests.

Dans tes notes : pourquoi vérifier `b.shape` toi-même, alors que NumPy lève déjà une erreur quand les formes sont incompatibles ? Relis f) de 🔮 16.11.

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="16")   # reload your saved file
with checked("16.13"):
    W_doc_13 = np.array([[1.0, 0.0, -1.0], [0.5, 1.0, 2.0]])
    print("docstring examples:", np.asarray(mylearn.nn.layers.dense_forward([[1.0, 2.0]], W_doc_13, np.array([0.0, 0.0, 1.0]))).tolist(),
          "and", np.asarray(mylearn.nn.layers.dense_forward([1.0, 2.0], W_doc_13)).tolist(),
          "(expected [[2.0, 2.0, 4.0]] and [2.0, 2.0, 3.0])")
with checked("16.13"):
    rng_13 = np.random.default_rng(1613)
    X_13, W_13, b_13 = X_STD[:50], rng_13.normal(0, 0.05, size=(784, 30)), rng_13.normal(size=30)
    z_13 = mylearn.nn.layers.dense_forward(X_13, W_13, b_13)
    if returned("16.13", "dense_forward", z_13):
        z_13 = np.asarray(z_13)
        verdict("16.13", z_13.shape == (50, 30) and z_13.dtype == np.float64,
                "50 images, 30 neurones : une sortie (50, 30) en float64.",
                f"attendu : un tableau (50, 30) en float64, pas {z_13.shape} en {z_13.dtype}.")
        if torch is not None:
            oracle_13 = torch.nn.functional.linear(torch.tensor(X_13), torch.tensor(W_13.T), torch.tensor(b_13)).numpy()
            verdict("16.13", z_13.shape == oracle_13.shape and np.allclose(z_13, oracle_13),
                    "mêmes valeurs que torch.nn.functional.linear (qui reçoit W.T : PyTorch range les poids en (n_out, n_in)).",
                    "tes valeurs diffèrent de torch.nn.functional.linear(x, W.T, b) : relis la convention z = x @ W + b.")
        one_by_one_13 = np.array([np.asarray(mylearn.nn.layers.dense_forward(x, W_13, b_13)) for x in X_13])
        verdict("16.13", one_by_one_13.shape == z_13.shape and np.allclose(one_by_one_13, z_13),
                "chaque image donne la même sortie seule (1-D) ou dans le mini-batch.",
                "une image seule (tableau 1-D) ne donne pas la même sortie que dans le batch : le biais est-il bien diffusé sur les lignes ?")
if written(lambda: mylearn.nn.layers.dense_forward(np.ones((1, 1)), np.ones((1, 1)))):
    run_mylearn_tests("layers", keyword="test_dense_forward_")
else:
    print("⏳ Ex 16.13 : les tests attendent ton dense_forward.")

## Partie B · Initialiser les poids, ordonner un graphe

Les dix méthodes d'initialisation de ta librairie (16.14), leurs lois dessinées avec leurs vraies densités (16.15), puis l'ordre d'évaluation d'un graphe (16.16).

### Ex 16.14 — Dix façons d'initialiser : init_weights 🔨 ★★ ⏱️ 25 min
**Objectif :** écrire les initialisations constante, uniforme, normale, LeCun, Glorot et He, avec un générateur reçu en argument.  
**Prérequis :** Ex 16.13 · ch. 2 (lois uniforme et normale) · fiche §16.4.1 (la table des trois règles) · **Fil rouge :** synthétique · **mylearn :** `nn/layers.py` · **Parcours :** R, M, C

> **mylearn** : dans `mon_travail/mylearn/nn/layers.py`, mêmes règles qu'en 16.12. Enregistre, puis relance la cellule de vérification.

Écris `init_weights`. **Contrat** (la docstring, plus ces précisions que vérifient les tests) : `init_weights(n_in, n_out, method="he_normal", rng=None, *, scale=0.05)` rend un tableau `float64` de forme `(n_in, n_out)` ; `method` est l'un des noms de la docstring (`"zeros"`, `"constant"`, `"uniform"`, `"normal"`, puis `"lecun_uniform"`, `"lecun_normal"`, `"glorot_uniform"`, `"glorot_normal"`, `"he_uniform"`, `"he_normal"`, et les alias `"xavier_uniform"` et `"xavier_normal"`), avec les bornes et les écarts-types de la table de la fiche ; `scale` est la valeur de `"constant"`, la borne de `"uniform"` et l'écart-type de `"normal"`, et n'est utilisé par aucune autre méthode ; les tirages normaux ne sont **pas** tronqués ; tous les tirages viennent du générateur `rng` (si `rng` vaut `None`, un nouveau `np.random.default_rng()`), qu'aucune graine fixée à l'intérieur ne remplace, et l'état global `np.random` n'est jamais touché ; `ValueError` pour une méthode inconnue (vérifiée en premier), une taille < 1 ou un booléen.

La vérification essaie les exemples de la docstring, affiche pour une couche de 600 entrées et 400 neurones l'écart-type obtenu par chaque méthode face à celui de sa règle, vérifie la reproductibilité, puis lance les tests.

Dans tes notes : pourquoi un tirage de 1 000 × 1 000 poids donne-t-il un écart-type à 2 % près, mais pas un tirage de 10 × 10 ? Pourquoi `init_weights` reçoit-elle un générateur, plutôt qu'une graine fixée à l'intérieur ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="16")   # reload your saved file
with checked("16.14"):
    W_doc_14 = mylearn.nn.layers.init_weights(100, 50, "glorot_uniform", rng=np.random.default_rng(0))
    print("docstring examples:", np.asarray(mylearn.nn.layers.init_weights(2, 3, "constant", scale=0.5)).tolist(),
          "· shape", np.shape(W_doc_14), "· inside the bound:", bool(np.abs(W_doc_14).max() <= np.sqrt(6 / 150)))
with checked("16.14"):
    n_in_14, n_out_14 = 600, 400
    W_first_14 = mylearn.nn.layers.init_weights(n_in_14, n_out_14, "he_normal", rng=np.random.default_rng(1614))
    if returned("16.14", "init_weights", W_first_14):
        targets_14 = {"lecun_uniform": np.sqrt(1 / n_in_14), "lecun_normal": np.sqrt(1 / n_in_14),
                      "glorot_uniform": np.sqrt(2 / (n_in_14 + n_out_14)), "glorot_normal": np.sqrt(2 / (n_in_14 + n_out_14)),
                      "he_uniform": np.sqrt(2 / n_in_14), "he_normal": np.sqrt(2 / n_in_14)}
        rows_14, all_ok_14 = [f"{'method':15} {'target std':>10} {'your std':>10} {'largest |w|':>12}"], True
        for method_14, target_14 in targets_14.items():
            W_14 = np.asarray(mylearn.nn.layers.init_weights(n_in_14, n_out_14, method_14, rng=np.random.default_rng(1614)))
            std_14 = float(W_14.std())
            ok_14 = W_14.shape == (n_in_14, n_out_14) and abs(std_14 / target_14 - 1) < 0.02
            if method_14.endswith("uniform"):
                ok_14 = ok_14 and np.abs(W_14).max() <= np.sqrt(3) * target_14
            all_ok_14 = all_ok_14 and ok_14
            rows_14.append(f"{method_14:15} {target_14:10.4f} {std_14:10.4f} {np.abs(W_14).max():12.4f} {'✅' if ok_14 else '❌'}")
        print("\n".join(rows_14))
        verdict("16.14", all_ok_14, "les six méthodes ont l'écart-type de leur règle (à 2 % près), les uniformes restent dans leur borne.",
                "une méthode n'a pas l'écart-type de sa règle (ligne ❌) : relis la table de la fiche (§16.4.1) ; "
                "rng.normal attend un écart-type, et U(−a, a) a l'écart-type a/√3.")
        same_14 = np.array_equal(mylearn.nn.layers.init_weights(5, 4, rng=np.random.default_rng(7)),
                                 mylearn.nn.layers.init_weights(5, 4, rng=np.random.default_rng(7)))
        shared_14 = np.random.default_rng(7)
        moved_14 = not np.array_equal(mylearn.nn.layers.init_weights(5, 4, rng=shared_14), mylearn.nn.layers.init_weights(5, 4, rng=shared_14))
        verdict("16.14", same_14 and moved_14, "même graine, mêmes poids ; un même générateur, deux appels, deux tirages différents.",
                "attendu : deux générateurs de même graine donnent les mêmes poids, et deux appels avec le MÊME générateur des "
                "poids différents (tires-tu avec rng, et seulement avec lui ?).")
if written(lambda: mylearn.nn.layers.init_weights(1, 1, "zeros")):
    run_mylearn_tests("layers", keyword="test_init_weights_")
else:
    print("⏳ Ex 16.14 : les tests attendent ton init_weights.")

### Ex 16.15 — Reproduire les figures 16.5 et 16.6, puis comparer Glorot et He 🎨 ★★ ⏱️ 20 min
**Objectif :** dessiner des lois d'initialisation avec leurs vraies densités, et voir ce qui sépare Glorot et He.  
**Prérequis :** Ex 16.14 · ch. 2 (densités) · fiche §16.4.1 (encadré sur les densités) · **Fil rouge :** synthétique · **Parcours :** C

Les figures 16.5 et 16.6 du livre montrent une loi uniforme sur $[-0{,}05 ;\ 0{,}05]$ et une loi normale presque nulle aux mêmes bornes, sans échelle verticale. Refais-les avec leurs **vraies** hauteurs, puis ajoute un troisième panneau.

Écris `draw_15(rng)`, qui dessine trois panneaux côte à côte et rend la figure (sans appeler `plt.show()`) :
1. l'histogramme (`density=True`) de 20 000 tirages de $\mathcal{U}(-0{,}05 ;\ 0{,}05)$, et sa densité théorique tracée par-dessus ;
2. la même chose pour $\mathcal{N}(0 ;\ (0{,}05/3)^2)$, la loi de la figure 16.6 (son écart-type est $0{,}05/3$) ;
3. les histogrammes des poids d'une couche de 512 × 512 initialisée avec `"glorot_normal"`, puis avec `"he_normal"`, sur les mêmes axes.

Tous les tirages passent par **ton** `init_weights`, avec le générateur `rng`. Calcule aussi, par la formule (pas en lisant la figure) :
a) `height_15` : la hauteur de la densité de $\mathcal{U}(-0{,}05 ;\ 0{,}05)$ ;
b) `peak_15` : la hauteur du pic de la densité de $\mathcal{N}(0 ;\ (0{,}05/3)^2)$ ;
c) `ratio_15` : pour une couche carrée ($n_{in} = n_{out}$), l'écart-type de He divisé par celui de Glorot.

Dans tes notes : la légende du livre parle de « probabilités » presque nulles en $\pm 0{,}05$ ; que vaut la probabilité qu'un poids de la loi normale dépasse 0,05 en valeur absolue ? Sur le troisième panneau, d'où vient l'écart entre les deux histogrammes, et lequel conviendrait à une couche ReLU ?

In [ ]:
def draw_15(rng):
    """Draw three panels with matplotlib (do not call plt.show()):
    (1) the histogram (density=True) of 20,000 draws of U(-0.05, 0.05) and its true density;
    (2) the same for N(0, (0.05/3)**2), the normal law of figure 16.6;
    (3) the histograms of the weights of a 512 x 512 layer initialised with "glorot_normal" and "he_normal".
    Use mylearn.nn.layers.init_weights for every draw, with the generator rng. Return the figure."""
    raise NotImplementedError("draw_15")


height_15 = ...   # a) the height of the density of U(-0.05, 0.05) (exact)
peak_15 = ...     # b) the height of the peak of the density of N(0, (0.05/3)**2) (exact formula, not read on the plot)
ratio_15 = ...    # c) for a square layer (n_in = n_out), the std of He divided by the std of Glorot (exact)

In [ ]:
wb.check("16.15a", height_15)
wb.check("16.15b", peak_15)
wb.check("16.15c", ratio_15)
with wb.attempt("16.15"):
    figure_15 = draw_15(np.random.default_rng(1615))
    if returned("16.15", "draw_15", figure_15):
        plt.show()

### Ex 16.16 — Ordre topologique d'un graphe : topological_order 🔨 ★★ ⏱️ 25 min
**Objectif :** écrire l'algorithme de Kahn, avec une règle déterministe parmi les nœuds prêts et la détection des cycles.  
**Prérequis :** ✏️ 16.4 · fiche §16.2 (encadré sur l'ordre topologique) · **mylearn :** `nn/layers.py` · **Parcours :** C

> **mylearn** : dans `mon_travail/mylearn/nn/layers.py`, mêmes règles qu'en 16.12. Enregistre, puis relance la cellule de vérification.

Écris `topological_order`. **Contrat** (la docstring, plus ces précisions que vérifient les tests) : `topological_order(graph)` reçoit un dictionnaire `{parent: [enfants, ...]}` (des noms en chaînes de caractères) et rend la **liste** de tous les nœuds, chacun une fois : ceux qui ne sont que des enfants (sans clé) et ceux qui n'ont ni parent ni enfant (une clé de liste vide) compris. Chaque parent vient avant tous ses enfants ; parmi les nœuds prêts (tous leurs parents déjà rangés), on range toujours le **plus petit** dans l'ordre alphabétique, si bien que l'ordre est unique (ce n'est donc pas un parcours couche par couche). `ValueError` si le graphe a un cycle, une arête d'un nœud vers lui-même comprise. La fonction ne modifie pas le dictionnaire reçu.

La vérification essaie les exemples de la docstring, contrôle sur 30 graphes au hasard que chaque nœud vient après ses parents, vérifie la détection d'un cycle, puis lance les tests (dont une comparaison avec `graphlib` de la bibliothèque standard).

Dans tes notes : combien d'opérations ton algorithme fait-il pour un graphe de $V$ nœuds et $E$ arêtes ? À quoi servira un ordre topologique au ch. 18, et dans quel sens le parcourra-t-on ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="16")   # reload your saved file
with checked("16.16"):
    print("docstring examples:", mylearn.nn.layers.topological_order({"x": ["h1", "h2"], "h1": ["y"], "h2": ["y"]}),
          "and", mylearn.nn.layers.topological_order({"b": ["c"], "a": ["c"]}),
          "(expected ['x', 'h1', 'h2', 'y'] and ['a', 'b', 'c'])")
with checked("16.16"):
    rng_16 = np.random.default_rng(1616)
    respected_16 = True
    for trial_16 in range(30):                       # random DAGs: an edge only from a smaller to a larger index
        names_16 = [f"n{k:02d}" for k in rng_16.permutation(12)]
        graph_16 = {names_16[i]: [names_16[j] for j in range(i + 1, 12) if rng_16.random() < 0.25] for i in range(12)}
        order_16 = mylearn.nn.layers.topological_order(graph_16)
        position_16 = {node: k for k, node in enumerate(order_16)}
        respected_16 = respected_16 and sorted(order_16) == sorted(names_16) and all(
            position_16[parent] < position_16[child] for parent, children in graph_16.items() for child in children)
    verdict("16.16", respected_16, "sur 30 graphes au hasard, chaque nœud apparaît une fois, après tous ses parents (comme l'exige graphlib).",
            "sur un graphe au hasard, un nœud manque, apparaît deux fois ou vient avant l'un de ses parents.")
    cycle_16 = {"a": ["b"], "b": ["c"], "c": ["a"], "d": ["a"]}
    try:
        mylearn.nn.layers.topological_order(cycle_16)
        print("❌ Ex 16.16 : le graphe a → b → c → a a un cycle : ta fonction doit lever une ValueError.")
    except ValueError:
        print("✅ Ex 16.16 : un graphe avec un cycle lève une ValueError, comme graphlib lève une CycleError.")
if written(lambda: mylearn.nn.layers.topological_order({"a": ["b"]})):
    run_mylearn_tests("layers", keyword="test_topological_order_")
else:
    print("⏳ Ex 16.16 : les tests attendent ton topological_order.")

## Partie C · Empiler les couches

Ton `init_mlp` et ton `mlp_forward` (16.17), les poids d'un `MLPClassifier` de scikit-learn relus par ta passe avant (16.18), des neurones jumeaux (16.19), une passe avant buggée (16.20), puis des tests de reproductibilité (16.21).

### Ex 16.17 — Empiler les couches : init_mlp et mlp_forward 🔨 ★★ ⏱️ 30 min
**Objectif :** initialiser un MLP couche par couche, puis écrire sa passe avant avec le cache que demandera la rétropropagation.  
**Prérequis :** Ex 16.13 · Ex 16.14 · fiche, section sur la passe avant (empiler les couches, encadré sur la non-linéarité) · **Fil rouge :** MNIST · **mylearn :** `nn/layers.py` · **Parcours :** R, M, C

> **mylearn** : dans `mon_travail/mylearn/nn/layers.py`, mêmes règles qu'en 16.12. Enregistre, puis relance la cellule de vérification.

Écris `init_mlp` et `mlp_forward`. **Contrat** (les docstrings, plus ces précisions que vérifient les tests) :
- `init_mlp(layer_sizes, method="he_normal", rng=None)` rend une **liste** de couples `(W, b)`, un par couche : `W = init_weights(n_(l-1), n_l, method, rng)` et `b = np.zeros(n_l)` ; **un seul** générateur (`np.random.default_rng(rng)`) sert à toutes les couches, tirées dans l'ordre (la première d'abord), si bien que deux couches de même forme ont des poids différents ; `ValueError` comme `count_parameters` et `init_weights` ;
- `mlp_forward(x, params, activation=None, output_activation=None, return_cache=False)` applique les couches dans l'ordre : `z = dense_forward(a, W, b)`, puis `activation(z)` pour une couche cachée et `output_activation(z)` pour la dernière (`None` : l'identité ; la sortie est alors faite des logits) ; avec `return_cache=True`, elle rend le couple `(out, cache)`, où `cache` est une liste d'un couple `(a_prev, z)` par couche, `a_prev` étant l'entrée de la couche (`x` pour la première) et `z` sa somme pondérée **avant** l'activation ; `ValueError` si `params` est vide, ou si les formes ne s'enchaînent pas (comme `dense_forward`). Elle ne modifie ni `x` ni les paramètres.

La vérification essaie les exemples des docstrings, initialise un MLP 784-128-64-10 en Glorot normale, compare ses logits sur 256 images à ceux d'un `torch.nn.Sequential` qui a reçu les mêmes poids, affiche l'écart-type de `z` couche par couche, puis lance les tests.

Dans tes notes : pourquoi le cache garde-t-il `z` et pas seulement l'activation ? Quelle est la forme de `W` dans `cache` et dans PyTorch, et pourquoi la cellule transpose-t-elle avant de copier ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="16")   # reload your saved file
with checked("16.17"):
    params_doc_17 = mylearn.nn.layers.init_mlp([2, 4, 4, 1], rng=np.random.default_rng(0))
    print("docstring example (init_mlp): shapes", [np.shape(W) for W, b in params_doc_17], "(expected [(2, 4), (4, 4), (4, 1)])",
          "· size", sum(np.size(W) + np.size(b) for W, b in params_doc_17), "(expected 37)")
with checked("16.17"):
    params_ex_17 = [(np.array([[1.0, -1.0], [0.5, -2.0]]), np.array([0.0, 1.0])), (np.array([[1.0], [1.0]]), np.array([-1.0]))]
    out_ex_17, cache_ex_17 = mylearn.nn.layers.mlp_forward(np.array([[1.0, 1.0]]), params_ex_17, activation=relu, return_cache=True)
    print("docstring example (mlp_forward):", np.asarray(out_ex_17).tolist(), "(expected [[0.5]]) · cache z of the hidden layer",
          np.asarray(cache_ex_17[0][1]).tolist(), "(expected [[1.5, -2.0]])")
with checked("16.17"):
    sizes_17 = [784, 128, 64, 10]
    params_17 = mylearn.nn.layers.init_mlp(sizes_17, "glorot_normal", rng=np.random.default_rng(1617))
    X_17 = X_STD[:256]
    logits_17, cache_17 = mylearn.nn.layers.mlp_forward(X_17, params_17, activation=np.tanh, return_cache=True)
    verdict("16.17", np.shape(logits_17) == (256, 10) and len(cache_17) == 3
            and [np.shape(a) for a, _ in cache_17] == [(256, 784), (256, 128), (256, 64)]
            and [np.shape(z) for _, z in cache_17] == [(256, 128), (256, 64), (256, 10)],
            "784-128-64-10 sur 256 images : des logits (256, 10) et un cache de 3 couples (a_prev, z) aux bonnes formes.",
            "attendu : des logits (256, 10) et un cache de 3 couples (a_prev, z), a_prev de formes (256, 784), (256, 128), "
            "(256, 64) et z de formes (256, 128), (256, 64), (256, 10).")
    if torch is not None:
        model_17 = torch.nn.Sequential(torch.nn.Linear(784, 128), torch.nn.Tanh(), torch.nn.Linear(128, 64), torch.nn.Tanh(),
                                       torch.nn.Linear(64, 10)).double()
        with torch.no_grad():
            for layer_17, (W_17, b_17) in zip(model_17[::2], params_17):
                layer_17.weight.copy_(torch.tensor(np.asarray(W_17).T))   # PyTorch stores (n_out, n_in)
                layer_17.bias.copy_(torch.tensor(np.asarray(b_17)))
            oracle_17 = model_17(torch.tensor(X_17)).numpy()
        verdict("16.17", np.shape(logits_17) == oracle_17.shape and np.allclose(logits_17, oracle_17),
                "mêmes logits qu'un torch.nn.Sequential de Linear et de Tanh qui a reçu tes poids (transposés).",
                "tes logits diffèrent de ceux de PyTorch avec les mêmes poids : l'activation est-elle appliquée aux couches "
                "cachées seulement, et z est-il gardé AVANT l'activation ?")
    stds_17 = [float(np.std(z)) for _, z in cache_17]
    print("std of z, layer by layer:", np.round(stds_17, 3).tolist())
if written(lambda: mylearn.nn.layers.init_mlp([1, 1])) or written(lambda: mylearn.nn.layers.mlp_forward(np.ones((1, 1)), [(np.ones((1, 1)), np.zeros(1))])):
    run_mylearn_tests("layers", keyword="test_init_mlp_ or test_mlp_forward_")
else:
    print("⏳ Ex 16.17 : les tests attendent ton init_mlp et ton mlp_forward.")

### Ex 16.18 — Relire un MLPClassifier de scikit-learn avec ton mlp_forward 📦 ★★ ⏱️ 25 min
**Objectif :** retrouver les prédictions d'un réseau de scikit-learn à partir de ses poids, avec ta propre passe avant.  
**Prérequis :** Ex 16.17 · ch. 15 (attributs appris) · fiche, encadré sur les poids d'un MLPClassifier · **Fil rouge :** MNIST · **Parcours :** C

La cellule fournie entraîne un `MLPClassifier(hidden_layer_sizes=(64,))` (une couche cachée de 64 neurones ReLU) pendant 30 epochs sur 6 000 images MNIST, dont les pixels sont divisés par 255, et définit `softmax_18`, une softmax de chaque ligne. Ses poids sont dans `coefs_` et `intercepts_`, au format de ta librairie.

Calcule :
a) `n_params_18` : le nombre de paramètres du réseau, compté à partir de `coefs_` et `intercepts_` ;
b) `shape_18` : la forme de `coefs_[0]`, entre guillemets ;
puis range ses paramètres dans `params_18`, une liste de couples `(W, b)`, et calcule avec **ton** `mlp_forward` les probabilités `proba_18` des 2 000 images de test (ReLU dans la couche cachée, `softmax_18` à la sortie) :
c) `agree_18` : la part des images de test où l'argmax de `proba_18` est la classe prédite par `CLF_18.predict` ;
d) `gap_18` : le plus grand écart absolu entre `proba_18` et `CLF_18.predict_proba` ;
e) `accuracy_18` : l'accuracy de test de tes prédictions (l'argmax de `proba_18`) ;
f) `tanh_agree_18` : la part de c) si ta passe avant utilise `np.tanh` au lieu de ReLU, avec les mêmes poids.

Dans tes notes : que t'apprend f) sur ce qu'il faut savoir d'un réseau, en plus de ses poids, pour refaire ses prédictions ? Pourquoi une softmax n'est-elle pas nécessaire pour retrouver `predict` ?

In [ ]:
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier

X_TRAIN_18, Y_TRAIN_18 = wb.datasets.load_mnist("train", n=6000, flatten=True, normalize=True, seed=1618)
X_TEST_18, Y_TEST_18 = wb.datasets.load_mnist("test", n=2000, flatten=True, normalize=True, seed=1618)
X_TRAIN_18, X_TEST_18 = X_TRAIN_18.astype(float), X_TEST_18.astype(float)   # pixels / 255, in float64
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)                       # 30 epochs only, on purpose
    CLF_18 = MLPClassifier(hidden_layer_sizes=(64,), max_iter=30, random_state=0).fit(X_TRAIN_18, Y_TRAIN_18)
print(f"MLPClassifier: {CLF_18.n_iter_} epochs, test accuracy {CLF_18.score(X_TEST_18, Y_TEST_18):.3f}")


def softmax_18(z):
    """Softmax of each row (stable: the largest value of the row is subtracted first)."""
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

In [ ]:
n_params_18 = ...     # a) the number of parameters of CLF_18, counted from its coefs_ and intercepts_
shape_18 = ...        # b) the shape of CLF_18.coefs_[0], between quotes, e.g. "(4, 7)"
params_18 = ...       # the list of (W, b) of CLF_18, in the format of mylearn.nn.layers.mlp_forward
proba_18 = ...        # your probabilities for X_TEST_18: YOUR mlp_forward with ReLU and softmax_18 at the output
agree_18 = ...        # c) the share of test images where the argmax of proba_18 equals CLF_18.predict(X_TEST_18)
gap_18 = ...          # d) the largest absolute difference between proba_18 and CLF_18.predict_proba(X_TEST_18)
accuracy_18 = ...     # e) the test accuracy of your predictions (argmax of proba_18) against Y_TEST_18
tanh_agree_18 = ...   # f) the share of c) if your mlp_forward uses np.tanh instead of ReLU (same weights)

In [ ]:
wb.check("16.18a", n_params_18)
wb.check("16.18b", shape_18)
wb.check("16.18c", agree_18)
wb.check("16.18d", gap_18)
wb.check("16.18e", accuracy_18)
wb.check("16.18f", tanh_agree_18)

### Ex 16.19 — Initialisation constante : les neurones restent-ils jumeaux ? 🔮 ★★ ⏱️ 20 min
**Objectif :** prévoir, avant de le voir, ce que devient une couche initialisée par une constante pendant l'entraînement.  
**Prérequis :** Ex 16.17 · ∂ 16.6 · ch. 5 (gradient numérique, descente de gradient) · fiche §16.4.1 · **Fil rouge :** synthétique · **Parcours :** M, C

Un réseau 2-4-1 (tanh dans la couche cachée, pas d'activation à la sortie) apprend un « ou exclusif » de 200 points (`X_19`, labels 0 et 1), en minimisant la MSE par 200 pas de descente de gradient (learning rate 0,5). Le gradient est calculé par **différences finies** (ch. 5, `numerical_gradient_19`), en appelant ta `mlp_forward` : aucune formule de rétropropagation, rien qui puisse traiter deux neurones différemment. `spread_19` mesure l'écart entre les quatre neurones cachés (le plus grand écart entre les poids entrants, le biais et le poids sortant d'un neurone et ceux du neurone 0). Trois départs : tous les paramètres à 0,5 ; une initialisation de Glorot normale ; tous à 0,5, plus un bruit d'écart-type $10^{-4}$ sur les poids entrants.

Prédis, **avant** d'exécuter quoi que ce soit :
a) `twins_19` : après 200 pas, les quatre neurones partis de 0,5 sont-ils encore identiques à $10^{-5}$ près (`spread_19 < 1e-5`) ? `True` ou `False` ;
b) `loss_19_choice` : la MSE finale du départ constant, comparée à celle du départ de Glorot : `"A"` moins de la moitié ; `"B"` entre la moitié et le double ; `"C"` plus du double ;
c) `noise_19` : après 200 pas, le départ constant bruité : `"A"` les neurones sont encore jumeaux à $10^{-3}$ près ; `"B"` ils se sont séparés, mais la MSE reste proche de celle du départ constant ; `"C"` ils se sont séparés, et la MSE est proche de celle du départ de Glorot.

Puis exécute l'expérience : elle entraîne les trois départs, affiche leurs MSE et leurs écarts avant et après, et vérifie tes prédictions.

In [ ]:
X_19, Y_19 = wb.synth.make_xor(n=200, noise=0.15, seed=1619)   # two classes in "exclusive or", labels 0 and 1
Y_19 = Y_19.astype(float)


def loss_19(params):
    """Mean squared error of a 2-4-1 network with tanh in the hidden layer (YOUR mlp_forward), on the 200 points."""
    out = mylearn.nn.layers.mlp_forward(X_19, params, activation=np.tanh)
    return float(np.mean((np.asarray(out)[:, 0] - Y_19) ** 2))


def numerical_gradient_19(params, h=1e-6):
    """Central differences (chapter 5): the derivative of loss_19 with respect to every weight and every bias."""
    grads = []
    for W, b in params:
        gradient = []
        for array in (W, b):
            g = np.zeros_like(array)
            for index in np.ndindex(array.shape):
                old = array[index]
                array[index] = old + h
                up = loss_19(params)
                array[index] = old - h
                down = loss_19(params)
                array[index] = old
                g[index] = (up - down) / (2 * h)
            gradient.append(g)
        grads.append(tuple(gradient))
    return grads


def train_19(params, steps=200, lr=0.5):
    """`steps` steps of gradient descent on a COPY of params; return the trained copy."""
    params = [(np.array(W, dtype=float), np.array(b, dtype=float)) for W, b in params]
    for _ in range(steps):
        for (W, b), (gW, gb) in zip(params, numerical_gradient_19(params)):
            W -= lr * gW
            b -= lr * gb
    return params


def spread_19(params):
    """How different the 4 hidden neurons are: the largest gap between the incoming weights, bias and outgoing weight
    of a neuron and those of neuron 0."""
    (W1, b1), (W2, _) = params
    neurons = np.vstack([W1, b1[None, :], W2.T])            # one column per hidden neuron
    return float(np.abs(neurons - neurons[:, :1]).max())


def constant_19(c):
    """Every weight and every bias equal to c."""
    return [(np.full((2, 4), c), np.full(4, c)), (np.full((4, 1), c), np.full(1, c))]

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
twins_19 = ...         # a) True or False
loss_19_choice = ...   # b) "A", "B" or "C"
noise_19 = ...         # c) "A", "B" or "C"

In [ ]:
if any(answer is ... or answer is None for answer in [twins_19, loss_19_choice, noise_19]):
    print("⏳ Ex 16.19 : écris d'abord tes trois prédictions, puis relance cette cellule.")
else:
    with wb.attempt("16.19"):
        rng_19 = np.random.default_rng(1619)
        starts_19 = {"constant 0.5": constant_19(0.5),
                     "random (glorot_normal)": mylearn.nn.layers.init_mlp([2, 4, 1], "glorot_normal", rng_19),
                     "constant 0.5 + noise 1e-4": [(constant_19(0.5)[0][0] + 1e-4 * rng_19.normal(size=(2, 4)), np.full(4, 0.5)),
                                                   (np.full((4, 1), 0.5), np.full(1, 0.5))]}
        results_19 = {}
        for name_19, start_19 in starts_19.items():
            trained_19 = train_19(start_19)
            results_19[name_19] = (loss_19(start_19), loss_19(trained_19), spread_19(start_19), spread_19(trained_19))
            print(f"{name_19:27} loss {results_19[name_19][0]:.4f} -> {results_19[name_19][1]:.4f} · "
                  f"spread of the neurons {results_19[name_19][2]:.2e} -> {results_19[name_19][3]:.2e}")
        print(f"a) constant start, spread_19 < 1e-5 after 200 steps: {results_19['constant 0.5'][3] < 1e-5}")
        wb.check("16.19a", twins_19)
        wb.check("16.19b", loss_19_choice)
        wb.check("16.19c", noise_19)

Dans tes notes : relie a) à la preuve ∂ 16.6. Un réseau dont les quatre neurones cachés sont jumeaux équivaut à un réseau à combien de neurones cachés, et peut-il apprendre un « ou exclusif » ? En c), d'où vient la séparation des neurones, et que faudrait-il pour que le réseau apprenne vraiment ?

### Ex 16.20 — Chasse au bug : transposée, biais mal diffusé, variance prise pour un écart-type 🐛 ★★ ⏱️ 25 min
**Objectif :** trouver trois erreurs qu'aucun message ne signale dans une initialisation et une passe avant écrites à la main.  
**Prérequis :** Ex 16.17 · 🔮 16.11 · fiche, section sur la passe avant (encadré sur le biais et la transposée), §16.4.1 · **Parcours :** C

Un collègue a écrit `init_20`, une initialisation de He normale, et `forward_20`, une passe avant ReLU. Il les a essayées sur un réseau 64-64-64-64 et un batch de 64 images : tout s'exécute sans erreur. Il y a pourtant **trois** erreurs : trouve-les et corrige-les, sans changer les signatures ni la convention du chapitre (`W` de forme `(n_in, n_out)`, un exemple par ligne).

La vérification contrôle trois propriétés, dans l'ordre (un point ne se teste que si les précédents passent) : 1) les poids d'une couche de 256 entrées ont l'écart-type de He ; 2) chaque image donne la même sortie seule ou dans le batch ; 3) la passe avant marche sur un réseau aux couches non carrées et donne le résultat d'une boucle sur les neurones. Elle affiche ensuite l'écart-type des sommes pondérées à travers 10 couches initialisées par `init_20`. Cet exercice se corrige avec `05_solutions.md`.

Dans tes notes : pour chaque erreur, pourquoi le réseau carré et le batch de 64 images la cachaient-ils ? Quel symptôme chacune, seule, aurait-elle sur un réseau profond ?

In [ ]:
# A colleague's initialisation and forward pass for square networks (64-64-64-64, ReLU), "tested" on a batch of
# 64 images: everything runs without an error, yet three bugs hide inside. Find and fix them (keep the signatures and
# the convention of the chapter: W of shape (n_in, n_out), one sample per row), then rerun the check cell.
def init_20(layer_sizes, rng):
    """He normal initialisation: a list of (W, b), W of shape (n_in, n_out), b of zeros."""
    params = []
    for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
        W = rng.normal(0.0, 2 / n_in, size=(n_in, n_out))
        params.append((W, np.zeros(n_out)))
    return params


def forward_20(x, params):
    """Forward pass with ReLU on the hidden layers and no activation at the output."""
    a = np.asarray(x, dtype=float)
    for k, (W, b) in enumerate(params):
        z = a @ W.T + b.reshape(-1, 1)
        a = z if k == len(params) - 1 else np.maximum(0.0, z)
    return a

In [ ]:
with checked("16.20"):
    rng_20 = np.random.default_rng(1620)
    params_20 = init_20([64, 64, 64, 64], rng_20)
    W_test_20 = np.asarray(init_20([256, 256], np.random.default_rng(0))[0][0])
    X_20 = X_STD[:64, :64]                                     # 64 images, 64 pixels: a square batch, on purpose
    trial_params_20 = [(np.asarray(W) + 0.0, np.random.default_rng(k).normal(size=np.shape(b))) for k, (W, b) in enumerate(params_20)]
    batch_20 = np.asarray(forward_20(X_20, trial_params_20))
    alone_20 = [np.asarray(forward_20(X_20[i:i + 1], trial_params_20)) for i in range(5)]
    small_20 = [(np.random.default_rng(3).normal(size=(64, 32)), np.random.default_rng(4).normal(size=32)),
                (np.random.default_rng(5).normal(size=(32, 10)), np.random.default_rng(6).normal(size=10))]
    try:
        rect_20 = np.asarray(forward_20(X_20, small_20))
    except ValueError:
        rect_20 = None
    loop_20 = X_20.copy()
    for k_20, (W_20, b_20) in enumerate(small_20):             # the same network, one neuron at a time
        loop_20 = np.array([[sum(row[j] * W_20[j, n] for j in range(W_20.shape[0])) + b_20[n] for n in range(W_20.shape[1])]
                            for row in loop_20])
        loop_20 = loop_20 if k_20 == len(small_20) - 1 else np.maximum(0.0, loop_20)
    diagnostics_20 = [                                         # (what is expected, verdict, the symptom seen)
        ("les poids d'une couche de 256 entrées ont l'écart-type de He, √(2/256) ≈ 0,088 (à 5 % près)",
         abs(float(W_test_20.std()) / np.sqrt(2 / 256) - 1) < 0.05,
         f" ; les tiens ont un écart-type de {fr(float(W_test_20.std()), 4)}"),
        ("chaque image donne la même sortie seule ou dans le batch (forme (1, 64) et mêmes valeurs)",
         all(a.shape == (1, 64) and np.allclose(a[0], batch_20[i]) for i, a in enumerate(alone_20)),
         f" ; une image seule donne un tableau de forme {alone_20[0].shape}" if alone_20[0].shape != (1, 64)
         else " ; une image seule ne donne pas les mêmes valeurs que dans le batch"),
        ("la passe avant marche sur un réseau aux couches non carrées (64-32-10) et donne le résultat d'une boucle sur les neurones",
         rect_20 is not None and rect_20.shape == (64, 10) and np.allclose(rect_20, loop_20),
         " ; sur ce réseau, ta passe avant lève ValueError" if rect_20 is None
         else f" ; elle rend un tableau de forme {rect_20.shape}" if rect_20.shape != (64, 10)
         else " ; ses valeurs diffèrent de celles de la boucle"),
    ]
    all_ok_20, failed_20 = True, None
    for k, (what, ok, seen) in enumerate(diagnostics_20, start=1):
        if not all_ok_20:
            print(f"⏳ Ex 16.20 : point {k} ({what}) : corrige d'abord le point {failed_20}.")
            continue
        verdict("16.20", ok, f"point {k} : {what}.", f"point {k} : attendu : {what}{seen}.")
        if not ok:
            all_ok_20, failed_20 = False, k
    deep_20 = init_20([64] + [64] * 10, np.random.default_rng(1620))
    stds_20, a_20 = [], X_20
    for W_20, b_20 in deep_20:
        z_20 = a_20 @ np.asarray(W_20) + b_20
        stds_20.append(float(z_20.std()))
        a_20 = np.maximum(0.0, z_20)
    print("std of z through 10 ReLU layers initialised by init_20:", np.round(stds_20, 3).tolist())

### Ex 16.21 — Rendre l'aléatoire reproductible et le prouver avec pytest 🛠️ ★★ ⏱️ 20 min
**Objectif :** écrire des tests pytest qui garantissent qu'une initialisation est reproductible et contrôlée par un générateur injecté.  
**Prérequis :** Ex 16.14 · 0A (pytest, `wb.run_pytest`) · fiche §16.4.1 (pièges classiques) · **Parcours :** C

Une initialisation est **reproductible** quand la même graine redonne les mêmes poids, et **contrôlée** quand c'est l'appelant qui choisit le hasard : il passe un générateur, que la fonction utilise et seulement lui. Écris au moins **trois** tests pytest pour une fonction `init_mlp(layer_sizes, rng=None)` qui a le contrat de `init_mlp_ok` (fournie) : une initialisation de He, dont toutes les couches sont tirées d'**un** générateur `np.random.default_rng(rng)`.

Range-les dans la liste `my_tests_21`. Comme en 2.31, la vérification lance vraiment pytest : tes tests doivent **passer** sur `init_mlp_ok` et **échouer** sur chacune des trois versions buggées fournies (lis-les : que fait chacune de travers ?). Tes tests ne peuvent utiliser que `init_mlp`, `np`, `math` et `pytest`. Cet exercice se corrige avec `05_solutions.md`.

Dans tes notes : pour chaque version buggée, le test qui l'attrape, et ce qui arriverait dans un vrai projet si elle passait inaperçue.

In [ ]:
def init_mlp_ok(layer_sizes, rng=None):
    """He normal initialisation of an MLP: a list of (W, b), all the layers drawn from ONE generator (correct)."""
    rng = np.random.default_rng(rng)
    return [(rng.normal(0.0, np.sqrt(2 / n_in), size=(n_in, n_out)), np.zeros(n_out))
            for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:])]


def init_mlp_bug_1(layer_sizes, rng=None):
    rng = np.random.default_rng(0)
    return [(rng.normal(0.0, np.sqrt(2 / n_in), size=(n_in, n_out)), np.zeros(n_out))
            for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:])]


def init_mlp_bug_2(layer_sizes, rng=None):
    return [(np.random.normal(0.0, np.sqrt(2 / n_in), size=(n_in, n_out)), np.zeros(n_out))
            for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:])]


def init_mlp_bug_3(layer_sizes, rng=None):
    seed = np.random.default_rng(rng).integers(2 ** 31)
    return [(np.random.default_rng(seed).normal(0.0, np.sqrt(2 / n_in), size=(n_in, n_out)), np.zeros(n_out))
            for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:])]

In [ ]:
# Write your test functions here (their names start with test_); they call init_mlp(layer_sizes, rng=None),
# whose contract is the one of init_mlp_ok. For example:
# def test_something():
#     params = init_mlp([3, 4], rng=np.random.default_rng(0))
#     assert ...

my_tests_21 = ...   # the list of your test functions

In [ ]:
if my_tests_21 is ...:
    print("⏳ Ex 16.21 : tests pas encore écrits.")
else:
    good_21 = wb.run_pytest(my_tests_21, subject=init_mlp_ok, name="init_mlp")
    if good_21.ok and good_21.passed < len(my_tests_21):
        print("❌ Ex 16.21 : pytest n'a pas trouvé tous tes tests : chaque fonction de test doit s'appeler test_…")
    verdict("16.21", good_21.ok and good_21.passed >= 3, f"tes tests passent sur la version juste ({good_21.passed} cas).",
            "tes tests doivent tous passer sur la version juste init_mlp_ok, et être au moins 3 (voir le détail au-dessus).")
    if good_21.ok:
        for bug_21 in [init_mlp_bug_1, init_mlp_bug_2, init_mlp_bug_3]:
            result_21 = wb.run_pytest(my_tests_21, subject=bug_21, name="init_mlp", quiet=True)
            verdict("16.21", result_21.failed + result_21.errors > 0, f"{bug_21.__name__} est attrapé ({result_21.failed} test(s) en échec).",
                    f"tes tests passent sur {bug_21.__name__}, qui est faux : lis-le, puis ajoute un test qui le fait échouer.")

## Partie D · Diagnostiquer les activations

Des histogrammes d'activations à lire (16.22), l'écart-type de 20 couches selon l'initialisation et l'activation (16.23), puis le défi d'un réseau ReLU de 50 couches nourri de pixels bruts (16.24).

### Ex 16.22 — Lire des histogrammes d'activations couche par couche 📈 ★★ ⏱️ 20 min
**Objectif :** reconnaître une initialisation à la forme des histogrammes de ses activations, et y lire l'évanouissement et la saturation.  
**Prérequis :** Ex 16.17 · fiche, section sur le diagnostic · **Fil rouge :** MNIST · **Parcours :** M

La figure fournie montre, pour 1 000 images MNIST standardisées, les histogrammes des activations tanh des six couches cachées d'un réseau 784-256-256-256-256-256-256-10 (biais nuls), pour trois initialisations rangées dans le désordre : les panneaux P, Q et R. Les trois initialisations sont $\mathcal{N}(0 ;\ 0{,}05^2)$ (une échelle fixe, comme l'exemple du livre), Glorot normale, et $\mathcal{N}(0 ;\ 1)$. Elle est tracée par la cellule fournie : tu n'as rien à écrire, seulement à lire.

Réponds en lisant la figure (pas en calculant) :
a) `panels_22` : les panneaux de $\mathcal{N}(0 ;\ 0{,}05^2)$, de Glorot et de $\mathcal{N}(0 ;\ 1)$, dans cet ordre (trois lettres collées, par exemple `"PQR"`) ;
b) `saturated_22` : vrai ou faux, avec $\mathcal{N}(0 ;\ 1)$, plus de la moitié des activations de la couche 6 sont saturées ($|a| > 0{,}99$) ;
c) `std_n005_22` : l'écart-type des activations de la couche 6 avec $\mathcal{N}(0 ;\ 0{,}05^2)$ : `"A"` moins de 0,05 ; `"B"` de 0,05 à 0,25 ; `"C"` de 0,25 à 0,6 ; `"D"` plus de 0,6 ;
d) `glorot_flat_22` : vrai ou faux, avec Glorot, l'écart-type des activations est le même de la couche 1 à la couche 6.

Dans tes notes : avec $\mathcal{N}(0 ;\ 1)$, pourquoi la couche 1 est-elle déjà saturée ? Avec $\mathcal{N}(0 ;\ 0{,}05^2)$, de quel facteur environ l'écart-type baisse-t-il d'une couche à l'autre, et pourquoi la règle de Glorot, conçue pour garder la variance, ne la garde-t-elle pas tout à fait avec tanh ?

In [ ]:
SIZES_22 = [784] + [256] * 6 + [10]          # six hidden layers of 256 tanh neurons
CANDIDATES_22 = [("normal", 0.05), ("glorot_normal", None), ("normal", 1.0)]
INITS_22 = dict(zip("PQR", [CANDIDATES_22[k] for k in np.random.default_rng(22).permutation(3)]))   # shuffled: read the figure


def activations_22(method, scale, X):
    """The tanh activations of the six hidden layers, for normal weights (zero biases): std `scale`, or the std of
    Glorot normal when scale is None."""
    rng = np.random.default_rng(1622)
    a, out = X, []
    for n_in, n_out in zip(SIZES_22[:-1], SIZES_22[1:]):
        std = np.sqrt(2 / (n_in + n_out)) if scale is None else scale
        a = np.tanh(a @ rng.normal(0.0, std, size=(n_in, n_out)))
        out.append(a)
    return out[:-1]                              # the six hidden layers (the 10 outputs are left out)


fig, axes = plt.subplots(3, 6, figsize=(15, 6.5), sharex=True)
for row, (panel, (method, scale)) in enumerate(INITS_22.items()):
    for layer, a in enumerate(activations_22(method, scale, X_STD[:1000])):
        axes[row, layer].hist(a.ravel(), bins=60, range=(-1, 1), density=True, color=f"C{row}")
        axes[row, layer].set_yticks([])
        if row == 0:
            axes[row, layer].set_title(f"hidden layer {layer + 1}")
    axes[row, 0].set_ylabel(f"panel {panel}", fontsize=12)
for ax in axes[-1]:
    ax.set_xlabel("activation")
fig.suptitle("tanh activations of a 784-256x6-10 network, 1,000 MNIST images, three initialisations")
fig.tight_layout()
plt.show()

In [ ]:
panels_22 = ...      # a) the panels of N(0, 0.05²), of Glorot normal and of N(0, 1), in this order: three letters
saturated_22 = ...   # b) True or False: with N(0, 1), more than half of the activations of layer 6 are saturated (|a| > 0.99)
std_n005_22 = ...    # c) the std of the layer-6 activations with N(0, 0.05²): "A" < 0.05, "B" 0.05 to 0.25,
                     #    "C" 0.25 to 0.6, "D" > 0.6
glorot_flat_22 = ... # d) True or False: with Glorot, the std of the activations stays the same from layer 1 to layer 6

In [ ]:
wb.check("16.22a", panels_22)
wb.check("16.22b", saturated_22)
wb.check("16.22c", std_n005_22)
wb.check("16.22d", glorot_flat_22)

### Ex 16.23 — Activations à travers 20 couches selon l'initialisation 🔬 ★★★ ⏱️ 45 min
**Objectif :** mesurer, couche par couche, l'écart-type des sommes pondérées de réseaux profonds pour quatre initialisations et deux activations.  
**Prérequis :** Ex 16.17 · Ex 16.14 · ∂ 16.7 · fiche, section sur le diagnostic · **Fil rouge :** MNIST · **Parcours :** M, C

Un réseau de 20 couches denses de 256 neurones (`SIZES_23`) reçoit 512 images MNIST standardisées (`X_23`). On fait varier son initialisation (`METHODS_23` : $\mathcal{N}(0 ;\ 0{,}05^2)$, LeCun, Glorot et He, toutes normales, biais nuls) et son activation (ReLU ou tanh, appliquée à **toutes** les couches).

Écris :
- `profile_23(activation, method, X, rng)` : initialise le réseau avec **ton** `init_weights` (avec `scale=0.05` pour `"normal"`), en tirant les couches dans l'ordre avec le générateur `rng`, lance **ton** `mlp_forward` sur `X` (l'activation passée à la fois en `activation` et en `output_activation`) et rend la liste des 20 écarts-types des sommes pondérées $z$, couche par couche (des `float` Python) ;
- `dead_23(X, rng)` : pour le réseau ReLU initialisé en He par **ton** `init_mlp`, la part des 256 neurones de la **dernière** couche dont la somme pondérée est $\le 0$ pour toutes les lignes de `X` (morts sur ce batch).

La vérification calcule les huit profils, les trace (échelle logarithmique), contrôle deux propriétés attendues en ReLU, puis vérifie que la part des neurones morts est entre 10 % et 50 %. Cet exercice se corrige avec `05_solutions.md`.

Dans tes notes : pour chaque activation, classe les quatre initialisations, et relie chaque pente à un gain de ∂ 16.7. En tanh, pourquoi He, de gain 2, garde-t-elle un écart-type plus grand que LeCun et Glorot, sans exploser ? La part des neurones morts dépend-elle de l'initialisation, à graine égale ? Pourquoi ?

In [ ]:
X_23 = X_STD[:512]                                    # 512 standardized images
SIZES_23 = [784] + [256] * 20                         # 20 dense layers of 256 neurons
ACTIVATIONS_23 = {"relu": relu, "tanh": np.tanh}
METHODS_23 = ["normal", "lecun_normal", "glorot_normal", "he_normal"]   # "normal": N(0, 0.05²), as in the book

In [ ]:
def profile_23(activation, method, X, rng):
    """Initialise a network of sizes SIZES_23 with YOUR init_weights (method; scale=0.05 for "normal") and zero
    biases, drawing the layers in order from the generator rng, run YOUR mlp_forward on X with `activation` on every
    layer (pass it as both activation and output_activation), and return the list of the 20 standard deviations of
    the pre-activations z, layer by layer (Python floats)."""
    raise NotImplementedError("profile_23")


def dead_23(X, rng):
    """For a ReLU network of sizes SIZES_23 initialised with "he_normal" (YOUR init_mlp), return the fraction of the
    256 neurons of the LAST layer whose pre-activation z is <= 0 for every row of X (dead on this batch)."""
    raise NotImplementedError("dead_23")

In [ ]:
with checked("16.23"):
    profiles_23 = {(name, method): profile_23(f, method, X_23, np.random.default_rng(1623))
                   for name, f in ACTIVATIONS_23.items() for method in METHODS_23}
    shapes_ok_23 = all(len(p) == 20 and all(isinstance(s, float) and s >= 0 for s in p) for p in profiles_23.values())
    verdict("16.23", shapes_ok_23, "huit profils de 20 écarts-types (des float Python positifs).",
            "chaque profil doit être une liste de 20 float Python positifs, un par couche.")
    if shapes_ok_23:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
        for ax, name in zip(axes, ACTIVATIONS_23):
            for method in METHODS_23:
                ax.plot(range(1, 21), profiles_23[(name, method)], marker="o", ms=3, label=method)
            ax.set_yscale("log")
            ax.set_title(f"{name}: std of z, layer by layer")
            ax.set_xlabel("layer")
            ax.legend(fontsize=8)
        axes[0].set_ylabel("std of z (log scale)")
        fig.tight_layout()
        plt.show()
        he_23, lecun_23 = profiles_23[("relu", "he_normal")], profiles_23[("relu", "lecun_normal")]
        verdict("16.23", 0.5 < he_23[-1] / he_23[0] < 2 and lecun_23[-1] / lecun_23[0] < 0.01,
                "ReLU : He garde l'écart-type sur 20 couches, LeCun le divise par plus de 100.",
                "attendu en ReLU : avec He, un rapport dernière/première couche entre 0,5 et 2 ; avec LeCun, sous 0,01. "
                "Tires-tu chaque couche avec le même générateur, et passes-tu ReLU à toutes les couches ?")
with checked("16.23"):
    dead_found_23 = dead_23(X_23, np.random.default_rng(1623))
    if returned("16.23", "dead_23", dead_found_23):
        print(f"ReLU + He, last layer: {float(dead_found_23):.1%} of the neurons are dead on these 512 images")
        verdict("16.23", 0.1 < float(dead_found_23) < 0.5,
                "une part de neurones morts entre 10 % et 50 %, comme attendu à cette profondeur (la valeur exacte dépend des tirages).",
                "attendu : une part (entre 0 et 1) de neurones morts entre 0,1 et 0,5 ; compte-t-on bien les colonnes de la "
                "dernière somme pondérée qui sont <= 0 sur TOUTES les lignes ?")

### Ex 16.24 — Défi : garder vivant un réseau ReLU de 50 couches 🏆 ★★★ ⏱️ 60 min
**Objectif :** trouver une initialisation qui garde vivant un réseau ReLU très profond, sans standardiser les données.  
**Prérequis :** Ex 16.23 · fiche, section sur le diagnostic (« Les données comptent aussi ») · **Fil rouge :** MNIST · **Parcours :** C

**Énoncé** : un réseau ReLU de 50 couches (`SIZES_24` : 784 pixels, 49 couches cachées de 128 neurones, 10 logits) reçoit des pixels **bruts** (de 0 à 255, sans standardisation). Écris `init_24(layer_sizes, X_sample, rng)`, qui rend ses paramètres, une liste de 50 couples `(W, b)` aux formes de la convention du chapitre, tels que, sur 1 000 images **de contrôle** que ta fonction ne voit pas (`X_HELD_24`), et pour chacune des trois graines de `SEEDS_24` : l'écart-type des sommes pondérées $z$ de **chaque** couche cachée, et celui des logits, soit compris entre 0,5 et 2 ; et moins de 10 % des neurones cachés soient morts (une somme pondérée $\le 0$ pour toutes les images de contrôle). Palier 🌟 : tous les écarts-types entre 0,8 et 1,25.

Règles : ta fonction peut regarder `X_sample` (1 000 autres images brutes) et tirer des nombres avec `rng`, et rien d'autre (ni `X_HELD_24`, ni l'état global `np.random`) ; tu peux utiliser tes fonctions de `mylearn.nn.layers`. Pour te situer : `init_mlp(SIZES_24, "he_normal", rng)` donne des écarts-types de 55 à 255 et 22 % à 27 % de neurones morts. La vérification mesure tout elle-même, en NumPy, sur les images de contrôle. Ce défi se corrige avec `05_solutions.md`.

Dans tes notes : ce que tu as essayé, dans l'ordre, et ce que chaque essai a changé aux trois mesures ; pourquoi régler seulement l'échelle de chaque couche ne suffit pas.

In [ ]:
X_RAW_24, _ = wb.datasets.load_mnist("train", n=3000, flatten=True, seed=1624)
X_RAW_24 = X_RAW_24.astype(float)                       # RAW pixels, from 0 to 255: no standardization
X_SAMPLE_24, X_HELD_24 = X_RAW_24[:1000], X_RAW_24[1000:2000]
SIZES_24 = [784] + [128] * 49 + [10]                    # 49 hidden ReLU layers of 128 neurons + 10 logits: 50 layers
SEEDS_24 = [0, 1, 2]


def measure_24(params, X):
    """(std of z of each hidden layer, std of the logits, fraction of hidden neurons dead on X), computed here with
    NumPy only: z <= 0 on every row of X means dead."""
    a, stds, dead = X, [], []
    for k, (W, b) in enumerate(params):
        z = a @ np.asarray(W, dtype=float) + np.asarray(b, dtype=float)
        stds.append(float(z.std()))
        if k < len(params) - 1:
            dead.append((z <= 0).all(axis=0))
            a = np.maximum(0.0, z)
    return stds[:-1], stds[-1], float(np.mean(np.concatenate(dead)))

In [ ]:
def init_24(layer_sizes, X_sample, rng):
    """Return a list of (W, b) for the ReLU network `layer_sizes` (W of shape (n_in, n_out), b of shape (n_out,)),
    chosen so that the network stays alive on raw MNIST pixels. You may look at X_sample (raw pixels, shape
    (1000, 784)) and draw random numbers from rng only."""
    raise NotImplementedError("init_24")

In [ ]:
with checked("16.24"):
    results_24 = {}
    for seed_24 in SEEDS_24:
        params_24 = init_24(SIZES_24, X_SAMPLE_24.copy(), np.random.default_rng(seed_24))
        if not returned("16.24", "init_24", params_24):
            break
        shapes_24 = [(np.shape(W), np.shape(b)) for W, b in params_24]
        expected_24 = [((a, b), (b,)) for a, b in zip(SIZES_24[:-1], SIZES_24[1:])]
        if shapes_24 != expected_24:
            print("❌ Ex 16.24 : init_24 doit rendre 50 couples (W, b), W de forme (n_in, n_out) et b de forme (n_out,).")
            break
        with np.errstate(all="ignore"):
            results_24[seed_24] = measure_24(params_24, X_HELD_24)
    if len(results_24) == len(SEEDS_24):
        for seed_24, (hidden_24, logits_24, dead_24) in results_24.items():
            print(f"seed {seed_24}: std of the hidden layers from {min(hidden_24):.3f} to {max(hidden_24):.3f} · "
                  f"std of the logits {logits_24:.3f} · dead neurons {dead_24:.1%}")
        lows_24 = [min(min(h), l) for h, l, _ in results_24.values()]
        highs_24 = [max(max(h), l) for h, l, _ in results_24.values()]
        deads_24 = [d for _, _, d in results_24.values()]
        ok_24 = all(np.isfinite(lows_24 + highs_24)) and min(lows_24) >= 0.5 and max(highs_24) <= 2 and max(deads_24) < 0.10
        star_24 = ok_24 and min(lows_24) >= 0.8 and max(highs_24) <= 1.25
        verdict("16.24", ok_24,
                "défi réussi : sur les 1 000 images de contrôle et les trois graines, chaque couche a un écart-type entre "
                "0,5 et 2, et moins de 10 % des neurones sont morts." + (" 🌟 Palier atteint : tous entre 0,8 et 1,25." if star_24 else ""),
                f"écarts-types de {fr(min(lows_24))} à {fr(max(highs_24))} (attendu : entre 0,5 et 2), neurones morts "
                f"jusqu'à {fr(100 * max(deads_24), 1)} % (attendu : moins de 10 %) : relis la section de la fiche sur le "
                "diagnostic, en particulier « Les données comptent aussi ».")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer à la main la passe avant d'un petit MLP, et donner la forme de chaque tableau d'une passe avant sur un mini-batch ?
2. Sais-tu compter de tête les paramètres et la mémoire d'un réseau dense ?
3. Sais-tu expliquer pourquoi on n'initialise pas les poids à une constante, choisir entre He et Glorot, et vérifier ce choix en mesurant les activations couche par couche ?

**Pour aller plus loin** : les articles de Glorot et Bengio (2010) et de He et coll. (2015), et l'initialisation LSUV de Mishkin et Matas (2016), cités dans la fiche. La suite : le ch. 17, qui ouvre la boîte noire des fonctions d'activation, puis le ch. 18, où le cache de ton `mlp_forward` servira à la rétropropagation.